# Data Cleaning

This notebook combines the NAV, investor transaction, and scheme performance cleaning workflows. It writes the cleaned datasets to `data/processed/` and handles invalid values, duplicates, date parsing, and business-day NAV gap filling.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists():
    BASE_DIR = BASE_DIR.parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
OUT_DIR = BASE_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {BASE_DIR}')

Repository: D:\Bluestock


In [2]:
# 1. Clean NAV history and forward-fill weekends/holidays on a business-day calendar
df = pd.read_csv(RAW_DIR / '02_nav_history.csv', dtype={'amfi_code': str})
n0 = len(df)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).sort_values(['amfi_code', 'date'])
df = df.drop_duplicates(subset=['amfi_code', 'date'])
df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
bad_nav = df['nav'].isna() | (df['nav'] <= 0)
print(f'Dropping {bad_nav.sum()} rows with invalid NAV')
df = df[~bad_nav]
filled = []
for code, group in df.groupby('amfi_code'):
    full_idx = pd.bdate_range(group['date'].min(), group['date'].max())
    group = group.set_index('date').reindex(full_idx).rename_axis('date').reset_index()
    group['amfi_code'] = code
    group['nav'] = group['nav'].ffill()
    filled.append(group)
df = pd.concat(filled, ignore_index=True)[['amfi_code', 'date', 'nav']]
df['daily_return_pct'] = df.groupby('amfi_code')['nav'].pct_change() * 100
df.to_csv(OUT_DIR / 'clean_nav.csv', index=False)
print(f'NAV rows: {n0} -> {len(df)}')

Dropping 0 rows with invalid NAV


NAV rows: 46000 -> 46000


In [3]:
# 2. Clean investor transactions
VALID_TYPES = {'sip': 'SIP', 'lumpsum': 'Lumpsum', 'lump sum': 'Lumpsum', 'redemption': 'Redemption'}
VALID_KYC = {'verified': 'Verified', 'pending': 'Pending'}
tx = pd.read_csv(RAW_DIR / '08_investor_transactions.csv', dtype={'amfi_code': str, 'investor_id': str})
n0 = len(tx)
tx['transaction_type'] = tx['transaction_type'].astype(str).str.strip().str.lower().map(VALID_TYPES)
tx = tx.dropna(subset=['transaction_type'])
tx['amount_inr'] = pd.to_numeric(tx['amount_inr'], errors='coerce')
tx = tx[tx['amount_inr'].notna() & (tx['amount_inr'] > 0)]
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], errors='coerce')
tx = tx.dropna(subset=['transaction_date'])
tx['kyc_status'] = tx['kyc_status'].astype(str).str.strip().str.lower().map(VALID_KYC).fillna('Pending')
tx = tx.drop_duplicates(subset=['investor_id', 'amfi_code', 'transaction_date', 'amount_inr', 'transaction_type'])
tx = tx.sort_values(['investor_id', 'transaction_date']).reset_index(drop=True)
tx.to_csv(OUT_DIR / 'clean_transactions.csv', index=False)
print(f'Transaction rows: {n0} -> {len(tx)}')

Transaction rows: 32778 -> 32778


In [4]:
# 3. Clean scheme performance metrics
NUMERIC_COLS = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'expense_ratio_pct']
perf = pd.read_csv(RAW_DIR / '07_scheme_performance.csv', dtype={'amfi_code': str})
n0 = len(perf)
for col in NUMERIC_COLS:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')
perf['expense_ratio_flag'] = ~perf['expense_ratio_pct'].between(0.1, 2.5)
perf['negative_sharpe_flag'] = perf['sharpe_ratio'] < 0
perf['bad_drawdown_flag'] = perf['max_drawdown_pct'] > 0
perf = perf.dropna(subset=['return_3yr_pct', 'sharpe_ratio', 'alpha'])
perf.to_csv(OUT_DIR / 'clean_performance.csv', index=False)
print(f'Performance rows: {n0} -> {len(perf)}')
print('Saved clean_nav.csv, clean_transactions.csv, and clean_performance.csv')

Performance rows: 40 -> 40
Saved clean_nav.csv, clean_transactions.csv, and clean_performance.csv
